# Band gap of a 2D material, with a verdict on whether to believe it

Upload a CIF or POSCAR of a **monolayer** (a single 2D layer with vacuum around it), or paste
its text, and this notebook returns:

- the band gap, in plain words: where the material starts absorbing light, what a gap like
  that is used for, and how far off such a prediction typically is;
- a **verdict** — *reliable*, *check*, or *out-of-domain* (do not use the number);
- the numbers behind it, for those who want them.

Nothing to install: press **Runtime → Run all**, then choose a file when asked. The model is
[NanoMatAI](https://github.com/ac1esan/nanomat-ai) (MIT licence); its predictions for
28 372 known structures can be browsed at <https://ac1esan.github.io/nanomat-ai/>.

In [ ]:
#@title 1. Set up (about a minute, once per session)
import os, subprocess, sys

REPO = os.environ.get("NANOMAT_ROOT", "/content/nanomat-ai")
if not os.path.exists(os.path.join(REPO, "nanomat")):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ac1esan/nanomat-ai", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric", "pymatgen"],
               check=True)
sys.path.insert(0, REPO)
from nanomat.llm_tools import predict_structure
print("ready")

In [ ]:
#@title 2. Your structure
# Paste the text of a POSCAR or CIF between the triple quotes - or leave it empty and you
# will be asked to upload one or more files. With neither, the bundled MoS2 example runs.
STRUCTURE_TEXT = """
"""

def inputs():
    text = STRUCTURE_TEXT.strip()
    if text:
        return [("pasted structure", text)]
    try:
        from google.colab import files          # only inside Colab
        uploaded = files.upload()
        if uploaded:
            out = []
            for name, data in uploaded.items():
                path = os.path.join("/content", name)
                with open(path, "wb") as f:
                    f.write(data)
                out.append((name, path))
            return out
    except ImportError:
        pass
    print("No structure given: running the bundled MoS2 example.")
    return [("MoS2 (example)", os.path.join(REPO, "examples", "MoS2.vasp"))]

STRUCTURES = inputs()

In [ ]:
#@title 3. Result
from IPython.display import Markdown, display

def show(name, r):
    if "error" in r:
        display(Markdown(f"### {name}\n**Could not read the structure.** {r['error']}"))
        return
    ood = r["verdict"].startswith("out-of-domain")
    gap = f"{r['band_gap_pbe_eV']:.2f} eV" + (" — **do not use this number**" if ood else "")
    lines = [f"### {r['formula']} · {name}",
             f"**In plain words.** {r['in_plain_words']}", "",
             "| | |", "|---|---|",
             f"| Verdict | **{r['verdict']}** |",
             f"| Band gap (PBE level) | {gap} |"]
    if not ood and r.get("gap_range90_eV"):
        lo, hi = r["gap_range90_eV"]
        lines.append(f"| 90% range | {lo:.2f} – {hi:.2f} eV |")
    if r.get("optical_gap_estimate_eV") is not None:
        lines.append(f"| Absorption onset (optical gap) | ≈ {r['optical_gap_estimate_eV']:.2f} eV, "
                     f"about {r['absorption_onset_nm']} nm ({r['absorption_onset_light']}) |")
    if r.get("quasiparticle_gap_eV") is not None:
        lines.append(f"| Quasiparticle gap (photoemission, transport) | ≈ {r['quasiparticle_gap_eV']:.2f} eV |")
    if r.get("gap_type"):
        lines.append(f"| Gap type | {r['gap_type']} |")
    if r.get("electron_affinity_eV") is not None:
        lines.append(f"| Band edges vs vacuum | {r['electron_affinity_eV']:.2f} / "
                     f"{r['ionisation_potential_eV']:.2f} eV |")
    for w in r.get("warnings", []):
        lines.append(f"\n⚠️ {w}")
    display(Markdown("\n".join(lines)))

for name, s in STRUCTURES:
    show(name, predict_structure(s))

**How to read it.** The model predicts the band gap at the level of a standard DFT
calculation (PBE), which runs below what experiments measure; the absorption onset and
the quasiparticle gap above are its estimates of the measurable quantities. The verdict
comes from three independent checks — a metal detector, the disagreement between five
models, and how far the structure is from anything the model was trained on — and
*out-of-domain* means the number should not be used at all.
[How the verdict works](https://github.com/ac1esan/nanomat-ai#knowing-when-not-to-believe-it).